# Deploy the Weather App to AWS: S3, CloudFront, Lambda, and CloudFormation

Run each cell in order with **Shift+Enter**. Lines starting with `!` run in the terminal. Values from one step (like the bucket name) are saved in Python variables and reused later, so there is no copying and pasting.

**The big picture:**
```
your files  ->  private S3 bucket  ->  CloudFront (HTTPS + security headers)  ->  browser
                                          |
                                          +-- /api/*  ->  Lambda (streams a Bedrock summary)
```
The page is a *static site*: three files (`index.html`, `styles.css`, `weather.js`) that run in the browser. The browser calls the weather services (Nominatim and the National Weather Service) directly. The AI summary comes from a small Lambda function, so AWS credentials never reach the browser.

**What gets created** (all by one CloudFormation stack, plus one small bucket for the Lambda zip):
- an **S3 bucket** that stores the three files and is closed to the public
- a **CloudFront distribution** that serves the files over HTTPS and sends `/api/*` to the Lambda function
- an **Origin Access Control** that lets only CloudFront read the bucket
- a **response headers policy** that adds browser security headers
- a **bucket policy** that enforces the two rules above
- a **Lambda function** with a streaming **function URL**, and an **IAM role** that can call one Bedrock model only

**Before you start:** in a terminal, run `aws sso login --profile <your-profile>`, set `AWS_PROFILE` to that profile in the same terminal, check it with `aws sts get-caller-identity`, then start Jupyter from this repo's top folder in that terminal. You also need Node.js 22+ and `npm` on the PATH, and access to your chosen model enabled in Bedrock in this region.

**Forecast AI summary:** the page calls `/api/summary`. CloudFront sends `/api/*` to the Lambda function and every other path to S3, so the browser code uses the same relative path locally and on AWS. Part 5 builds and uploads the function code. Part 6 deploys it.

## Part 1: Check who you are signed in as

**Why:** everything below is created in this account. The output shows the **account** (the 12-digit ID), and the **Arn**, which names the role you signed in with.

If this fails with an expired token, run `aws sso login` again in your terminal and restart Jupyter from that terminal.

In [ ]:
!aws sts get-caller-identity

Your profile also sets the **region**, which is where the stack and the bucket are created. CloudFront itself is global, but you will find the stack and bucket in the console under this region.

In [ ]:
!aws configure get region

## Part 2: Settings and a quick check

`STACK` is the name of the CloudFormation stack. `TEMPLATE` is the file that describes the resources.

The check makes sure the page styles live in `styles.css` and not inside `index.html`. That matters because the security headers (Part 3) block styles written inline in the page. It also confirms Jupyter was started from the repo folder, since the paths below are relative.

In [ ]:
import os
STACK = "weather-app"
TEMPLATE = "infra/weather-app.yaml"

ready = True
if not os.path.exists(TEMPLATE):
    print("Can't find", TEMPLATE, "- start Jupyter from the repo's top folder.")
    ready = False
if not os.path.exists("styles.css"):
    print("styles.css is missing. Move the CSS out of index.html first.")
    ready = False
html = open("index.html", encoding="utf-8").read()
if "<style" in html or " style=" in html:
    print("index.html still has inline styles. Move them to styles.css first.")
    ready = False
if ready:
    print("Ready: template found, styles.css exists, and index.html has no inline styles.")

## Part 3: Read the template

**Why:** CloudFormation is *infrastructure as code*. Instead of clicking through the console, you describe the resources in a text file and AWS builds exactly that. The file lives in git, so changes are reviewed like any other code, and the same file can build the site again in any account.

What to look for:
- **SiteBucket**: the S3 bucket. *Block Public Access* is on, so no one can make it public by accident. Files are encrypted at rest (*SSE-S3*, keys managed by S3). *Versioning* keeps old copies of each file so a bad upload can be undone.
- **SiteOAC**: the *Origin Access Control*. An **origin** is where CloudFront gets the files. OAC makes CloudFront sign each request to the bucket so the bucket can tell it really is CloudFront.
- **SecurityHeadersPolicy**: headers CloudFront adds to every response.
  - **CSP** (Content-Security-Policy) tells the browser which sources it may load scripts and styles from and which sites the page may call. Here: only this site, plus the two weather services. Anything else is blocked, which limits the damage if someone injects code into the page.
  - **HSTS** (Strict-Transport-Security) tells the browser to use only HTTPS for this site from now on.
  - The others stop the browser from guessing file types (`nosniff`), stop other sites from framing this page (`DENY`), and limit what address is sent to other sites (`Referrer-Policy`).
- **Distribution**: the CloudFront distribution. It redirects HTTP to HTTPS, serves `index.html` for `/`, compresses files, and uses edge locations in North America and Europe only (the cheapest price class).
- **SiteBucketPolicy**: lets only *this* distribution read the files, and refuses any request that is not encrypted.
- **Outputs**: values the stack hands back after it is built, such as the site's address.

Notice there is no account number, region, or bucket name in the file. CloudFormation fills those in when it runs.

In [ ]:
!type infra\weather-app.yaml

## Part 4: Validate the template

**Why:** catch typos before building anything. This sends the file to CloudFormation for a format check and prints the template's description. It creates nothing.

It only checks the format. A wrong setting value would still show up as an error during the deploy in Part 6.

In [ ]:
!aws cloudformation validate-template --template-body file://{TEMPLATE} --query Description --output text

## Part 5: Build and upload the Summary_Service

**Why:** Lambda runs code from a zip file in S3. This part:
1. reads the Bedrock model ID (`BEDROCK_MODEL_ID`). The stack's IAM role allows `bedrock:InvokeModelWithResponseStream` on this model only. Use an ID your account can call in this region. Models that need a cross-region inference profile use the profile ID, for example `us.anthropic.claude-...`.
2. builds the zip (`summary-lambda.mjs`, `summary-core.mjs`, and production `node_modules`) in `build/summary`.
3. creates a small private bucket for the zip, outside the stack (CloudFormation needs the zip to exist before it creates the function), and uploads it. The file name contains a hash of the contents, so a code change causes a function update on the next deploy.

Set `BEDROCK_MODEL_ID` in your environment before starting Jupyter, or edit the default here. The cell stops if it is empty.

In [ ]:
import hashlib, os, re, shutil, subprocess, zipfile

BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "").strip()
if not BEDROCK_MODEL_ID:
    raise SystemExit("Set BEDROCK_MODEL_ID (environment variable or this cell) before deploying.")

# Build the zip
BUILD = os.path.join("build", "summary")
shutil.rmtree("build", ignore_errors=True)
os.makedirs(BUILD)
for name in ["summary-lambda.mjs", "summary-core.mjs", "package.json", "package-lock.json"]:
    shutil.copy(name, BUILD)
subprocess.run("npm ci --omit=dev --no-audit --no-fund", shell=True, cwd=BUILD, check=True)

ZIP = os.path.join("build", "summary.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(BUILD):
        for f in sorted(files):
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, BUILD))
digest = hashlib.sha256(open(ZIP, "rb").read()).hexdigest()[:12]
CODE_KEY = f"summary-{digest}.zip"

# Small private bucket for the zip (not part of the stack)
account = subprocess.run("aws sts get-caller-identity --query Account --output text", shell=True, capture_output=True, text=True, check=True).stdout.strip()
region = subprocess.run("aws configure get region", shell=True, capture_output=True, text=True, check=True).stdout.strip()
CODE_BUCKET = f"{STACK}-code-{account}-{region}"
if subprocess.run(f"aws s3api head-bucket --bucket {CODE_BUCKET}", shell=True, capture_output=True).returncode != 0:
    subprocess.run(f"aws s3 mb s3://{CODE_BUCKET}", shell=True, check=True)
subprocess.run(f"aws s3 cp {ZIP} s3://{CODE_BUCKET}/{CODE_KEY}", shell=True, check=True)
print("Bedrock model:", BEDROCK_MODEL_ID)
print("Code uploaded: s3://" + CODE_BUCKET + "/" + CODE_KEY)

## Part 6: Create or update the stack

**Why:** a **stack** is the group of resources CloudFormation builds from one template. `deploy` works out what to create or change (a *change set*) and applies it:
- the first time, it creates everything
- after you edit the template or the function code, it changes only what is different
- if nothing changed, `--no-fail-on-empty-changeset` lets it finish quietly instead of reporting an error

`--tags Project=weather-app` labels the stack and its resources so you can find them and see their cost.

**This is slow and mostly silent.** The first run takes about 5 minutes, mostly waiting for CloudFront. That is normal.

If the first run fails, the stack is left in `ROLLBACK_COMPLETE` and can't be updated. Run the cleanup cells (Part 13 and Part 14), fix the problem, then run Part 5 and this cell again.

`--parameter-overrides` passes the zip location and `BedrockModelId` from Part 5. `--capabilities CAPABILITY_IAM` confirms that the stack may create the function's IAM role.

In [ ]:
!aws cloudformation deploy --stack-name {STACK} --template-file {TEMPLATE} --tags Project=weather-app --no-fail-on-empty-changeset --capabilities CAPABILITY_IAM --parameter-overrides SummaryCodeBucket={CODE_BUCKET} SummaryCodeKey={CODE_KEY} BedrockModelId={BEDROCK_MODEL_ID}

## Part 7: Read the stack outputs

**Why:** CloudFormation generated the bucket name and CloudFront picked the site address. Reading them from the stack's outputs means nothing is typed by hand.

In [ ]:
out = !aws cloudformation describe-stacks --stack-name {STACK} --query "Stacks[0].Outputs[?OutputKey=='BucketName'].OutputValue" --output text
BUCKET = out.s.strip()
out = !aws cloudformation describe-stacks --stack-name {STACK} --query "Stacks[0].Outputs[?OutputKey=='DistributionId'].OutputValue" --output text
DIST_ID = out.s.strip()
out = !aws cloudformation describe-stacks --stack-name {STACK} --query "Stacks[0].Outputs[?OutputKey=='DistributionDomainName'].OutputValue" --output text
DIST_DOMAIN = out.s.strip()
out = !aws cloudformation describe-stacks --stack-name {STACK} --query "Stacks[0].Outputs[?OutputKey=='BucketRegionalDomainName'].OutputValue" --output text
BUCKET_DOMAIN = out.s.strip()
SITE_URL = "https://" + DIST_DOMAIN

print("Bucket:         ", BUCKET)
print("Distribution ID:", DIST_ID)
print("Site URL:       ", SITE_URL)

## Part 8: Upload the app files

**Why:** the stack built an empty bucket. Now copy in the three files, and only those three. Each upload sets two things:
- `--content-type` tells the browser what kind of file it is (page, stylesheet, script). With the `nosniff` header on, the browser will refuse a script that is not labeled as one.
- `--cache-control` controls how long copies are kept:
  - `max-age=0` tells **browsers** to check for a newer version on every visit. If nothing changed, the check is a quick "not modified" reply.
  - `s-maxage=86400` tells **CloudFront edge locations** they may keep the file for one day (86,400 seconds), so most requests never reach S3.

Run Parts 7 to 9 again whenever you change the app files.

In [ ]:
!aws s3 cp index.html s3://{BUCKET}/index.html --content-type "text/html; charset=utf-8" --cache-control "max-age=0, s-maxage=86400"
!aws s3 cp styles.css s3://{BUCKET}/styles.css --content-type "text/css; charset=utf-8" --cache-control "max-age=0, s-maxage=86400"
!aws s3 cp weather.js s3://{BUCKET}/weather.js --content-type "application/javascript; charset=utf-8" --cache-control "max-age=0, s-maxage=86400"

## Part 9: Clear CloudFront's copies

**Why:** edge locations may still hold yesterday's files for up to a day. An **invalidation** tells every edge location to drop its copies, so the next request fetches the new files from S3. `/*` means every file and counts as one path. The first 1,000 paths each month are free.

The wait usually takes 1 to 2 minutes and is silent.

In [ ]:
out = !aws cloudfront create-invalidation --distribution-id {DIST_ID} --paths "/*" --query Invalidation.Id --output text
INV_ID = out.s.strip()
print("Invalidation:", INV_ID, "- waiting for it to finish...")
!aws cloudfront wait invalidation-completed --distribution-id {DIST_ID} --id {INV_ID}
print("Done. The site is serving the new files.")

## Part 10: Check the site

`curl -sI` asks for a page and prints only the response headers. Three checks:

**1. The site over HTTPS.** Expect `HTTP/1.1 200 OK` (or `HTTP/2 200`), and the security headers from Part 3: `content-security-policy`, `strict-transport-security`, `x-content-type-options`, `x-frame-options`, and `referrer-policy`. `x-cache` shows whether the edge location had the file (`Hit`) or fetched it from S3 (`Miss`).

In [ ]:
print(SITE_URL)
!curl -sI {SITE_URL}

**2. Plain HTTP.** Expect `301 Moved Permanently` with a `Location` that starts with `https://`. CloudFront sends the browser to the secure address.

In [ ]:
!curl -sI http://{DIST_DOMAIN}/

**3. Going around CloudFront, straight to S3.** Expect `403 Forbidden`. The bucket is private, and only requests signed by this distribution get in.

In [ ]:
!curl -sI https://{BUCKET_DOMAIN}/index.html

**4. The summary path through CloudFront.** `/api/summary` accepts only `POST`, so a `GET` should come back from the Lambda function as `405` with `Allow: POST`, not as the S3 `403` or the app page. That shows `/api/*` reaches the function and every other path still reaches S3. If you see the S3 answer, the `/api/*` behavior is not routed to the function.

In [ ]:
!curl -sI {SITE_URL}/api/summary

**5. A real summary.** This sends one small forecast to `/api/summary` and prints the streamed text. Expect a short paragraph. A `503` means Bedrock refused the call (check that the model is enabled for your account and matches `BedrockModelId`). A `502` means the function or Bedrock failed. See the log group `/aws/lambda/<function name>` in CloudWatch.

In [ ]:
!curl -s -X POST {SITE_URL}/api/summary -H "Content-Type: application/json" -d "{{\"city\":\"Test\",\"periods\":[{{\"name\":\"Today\",\"temperature\":70,\"shortForecast\":\"Sunny\",\"windSpeed\":\"5 mph\",\"windDirection\":\"N\"}}]}}"

## Part 11: Try it in the browser

The cell below opens the site. Then:
1. Enter a U.S. city and press **Get Weather**. The forecast should look the same as it does on `localhost:8080`, styles included.
2. Open the browser's developer tools (F12) and look at the **Console** tab. There should be no `Content-Security-Policy` errors. If there were, the CSP would be blocking something the app needs.
3. A `403` for `favicon.ico` is expected and harmless. The site has no icon file, and the bucket answers "forbidden" for files that don't exist.

In [ ]:
import webbrowser
webbrowser.open(SITE_URL)
print(SITE_URL)

## Part 12: What was built, and where to see it in the console

This lists every resource in the stack. *LogicalResourceId* is the name in the template, *ResourceType* is the kind of AWS resource.

In [ ]:
!aws cloudformation describe-stack-resources --stack-name {STACK} --query "StackResources[].[LogicalResourceId,ResourceType,ResourceStatus]" --output table

Where to find each one in the AWS console. CloudFormation and S3 are shown per region (use the region from Part 1). CloudFront is global.

| What | Console path | What to show |
|---|---|---|
| Stack | **CloudFormation > Stacks > weather-app** | *Resources* tab (the list above), *Outputs* tab (the site URL), *Events* tab (the build history), *Template* tab |
| Bucket | **S3 > Buckets >** the bucket name from Part 7 | *Objects* (the three files; turn on *Show versions*), *Permissions* (Block Public Access is on, bucket policy names the distribution), *Properties* (versioning and encryption) |
| Distribution | **CloudFront > Distributions >** the ID from Part 7 | *Origins* (the bucket, with the OAC), *Behaviors* (redirect to HTTPS, cache policy, response headers policy), *Invalidations* (from Part 9) |
| Origin access | **CloudFront > Security > Origin access** | `weather-app-oac` |
| Headers policy | **CloudFront > Policies > Response headers** (Custom) | `weather-app-security-headers` |

## Part 13: Cleanup, step 1: empty the bucket

**Why:** CloudFormation will not delete a bucket that still has files in it. Because versioning is on, deleting a file normally just hides it behind a *delete marker* and keeps the old versions. So this cell deletes every version and every delete marker, one at a time.

It finds the bucket from the stack, so it works even after restarting Jupyter, and it is safe to run again.

In [ ]:
import subprocess
STACK = "weather-app"

if subprocess.run(f"aws cloudformation describe-stacks --stack-name {STACK}", shell=True, capture_output=True).returncode != 0:
    print(f"No stack named {STACK}. Nothing to delete.")
else:
    out = !aws cloudformation describe-stack-resources --stack-name {STACK} --logical-resource-id SiteBucket --query "StackResources[0].PhysicalResourceId" --output text
    BUCKET = out.s.strip()
    if BUCKET in ["", "None"] or subprocess.run(f"aws s3api head-bucket --bucket {BUCKET}", shell=True, capture_output=True).returncode != 0:
        print("The stack has no bucket left. Nothing to empty.")
    else:
        for kind in ["Versions", "DeleteMarkers"]:
            out = !aws s3api list-object-versions --bucket {BUCKET} --query "{kind}[].[Key,VersionId]" --output text
            for line in out:
                if line.strip() in ["", "None"]:
                    continue
                key, version = line.split()
                !aws s3api delete-object --bucket {BUCKET} --key {key} --version-id {version} --query VersionId --output text
        print("Bucket", BUCKET, "is empty.")

## Part 14: Cleanup, step 2: delete the stack

**Why:** deleting the stack deletes everything it created (this cell also removes the small code bucket from Part 5), in the right order. Run Part 13 first, or the bucket delete fails.

**This is slow and silent:** about 5 to 15 minutes, because CloudFront first turns the distribution off at every edge location before deleting it.

In [ ]:
import subprocess
STACK = "weather-app"

if subprocess.run(f"aws cloudformation describe-stacks --stack-name {STACK}", shell=True, capture_output=True).returncode != 0:
    print(f"No stack named {STACK}. Nothing to delete.")
else:
    !aws cloudformation delete-stack --stack-name {STACK}
    print("Deleting the stack. This takes 5 to 15 minutes...")
    !aws cloudformation wait stack-delete-complete --stack-name {STACK}
    if subprocess.run(f"aws cloudformation describe-stacks --stack-name {STACK}", shell=True, capture_output=True).returncode != 0:
        acct = subprocess.run("aws sts get-caller-identity --query Account --output text", shell=True, capture_output=True, text=True).stdout.strip()
        reg = subprocess.run("aws configure get region", shell=True, capture_output=True, text=True).stdout.strip()
        code_bucket = f"{STACK}-code-{acct}-{reg}"
        if subprocess.run(f"aws s3api head-bucket --bucket {code_bucket}", shell=True, capture_output=True).returncode == 0:
            !aws s3 rb s3://{code_bucket} --force
        print("Stack and code bucket deleted. Nothing from this demo is left running.")
    else:
        out = !aws cloudformation describe-stacks --stack-name {STACK} --query "Stacks[0].StackStatus" --output text
        print("Stack status:", out.s.strip(), "- check the Events tab in the CloudFormation console.")

## Before you commit: clear the saved outputs

Saved outputs include your AWS account ID and role name, so they must not go into git. Run the cell below, then **close this notebook without saving** (choose Don't Save or Revert). If you save, the editor writes the old outputs back. Reopen the notebook to check that every cell is empty.

In [ ]:
import json
path = "deploy-aws.ipynb"
with open(path, encoding="utf-8") as f:
    nb = json.load(f)
for cell in nb["cells"]:
    if cell["cell_type"] == "code":
        cell["outputs"] = []
        cell["execution_count"] = None
with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)
print("Outputs cleared in the file. Close this notebook WITHOUT saving, then reopen it.")